In [ ]:
# Cell 1 — Imports
from pathlib import Path
from datetime import datetime
import numpy as np
import cv2
import toml
import msgpack
import msgpack_numpy as m
m.patch()
print("Imports OK")


In [ ]:
# Cell 2 — Paths (update per recording)
repo_root          = Path("E:/Ragav/MS Bio Engineering/NOARK_backbone")
CAM_CALIB_PATH     = repo_root / "notebooks" / "calibration" / "output" / "good.toml"
TABLE_CALIB_OUTPUT = repo_root / "estimator" / "charuco_pose" / "charuco_pose.toml"
print(f"CAM_CALIB_PATH     : {CAM_CALIB_PATH}  | Exists: {CAM_CALIB_PATH.exists()}")
print(f"TABLE_CALIB_OUTPUT : {TABLE_CALIB_OUTPUT}  | Exists: {TABLE_CALIB_OUTPUT.exists()}")
# Table-frame recording (run Cell 4 once per physical setup)
TABLE_FRAMES_MSGPACK = Path("E:/data/noark_data_may_15/table_frame_may_15/webcam_color.msgpack")
TABLE_TS_MSGPACK     = Path("E:/data/noark_data_may_15/table_frame_may_15/webcam_timestamp.msgpack")

# Experiment recording (run Cell 5 per experiment)
RECORDING_DIR      = Path("E:/data/noark_data_may_15/rotation_t1_may_15")
FRAMES_MSGPACK     = RECORDING_DIR / "webcam_color.msgpack"
TIMESTAMPS_MSGPACK = RECORDING_DIR / "webcam_timestamp.msgpack"
CAM_CSV_OUT        = RECORDING_DIR / "camera_data.csv"

for p in [CAM_CALIB_PATH, TABLE_FRAMES_MSGPACK, FRAMES_MSGPACK, TIMESTAMPS_MSGPACK]:
    print(f"{'OK' if p.exists() else 'MISSING'}  {p}")


In [ ]:
# Cell 3 — Camera calibration + undistortion maps + ChArUco detector
# Must run before Cell 4 and Cell 5
calib      = toml.load(CAM_CALIB_PATH)
cam_K      = np.array(calib["calibration"]["camera_matrix"])
cam_dist   = np.array(calib["calibration"]["dist_coeffs"]).flatten()[:4].reshape(4,1)
resolution = tuple(calib["camera"]["resolution"])
WIDTH, HEIGHT = resolution

new_K = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
    cam_K, cam_dist, resolution, np.eye(3), balance=1.0
)
map1, map2 = cv2.fisheye.initUndistortRectifyMap(
    cam_K, cam_dist, np.eye(3), new_K, resolution, cv2.CV_16SC2
)

# ChArUco board — mirrors charuco_estimator.py defaults
SQUARES_X, SQUARES_Y = 4, 3
SQUARE_LENGTH = 0.037
MARKER_LENGTH = 0.027
# ChArUco board uses 4x4 dict (for table calibration in Cell 4)
charuco_dict     = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
board            = cv2.aruco.CharucoBoard(
    (SQUARES_X, SQUARES_Y), SQUARE_LENGTH, MARKER_LENGTH, charuco_dict
)
charuco_detector = cv2.aruco.CharucoDetector(board)

# NOARK markers use AprilTag dict (for experiment detection in Cell 5)
apriltag_dict  = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_APRILTAG_36h11)
aruco_params   = cv2.aruco.DetectorParameters()
aruco_params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_CONTOUR
aruco_detector = cv2.aruco.ArucoDetector(apriltag_dict, aruco_params)
# NOARK marker config (from camera_pose.py)
MARKER_IDS     = [12, 14, 20]
MARKER_LENGTH_NOARK = 0.05
MARKER_OFFSETS = {
    12: np.array([ 0.000, 0, -0.055]),
    14: np.array([-0.126, 0, -0.054]),
    20: np.array([ 0.126, 0, -0.054]),
}

print(f"Resolution : {resolution}")
print(f"Board      : {SQUARES_X}x{SQUARES_Y}  square={SQUARE_LENGTH*100:.1f}cm  marker={MARKER_LENGTH*100:.1f}cm")
print("Ready")


In [ ]:
# Cell 4 — TABLE CALIBRATION  (run ONCE per physical setup)
# Processes table-frame msgpack -> estimates ChArUco board pose -> saves charuco_pose.toml
# Mirrors charuco_estimator.py::process_frame() + save_calibration_data()

print("Loading table frames...")
with open(TABLE_FRAMES_MSGPACK, "rb") as f:
    table_frames = [item for item in msgpack.Unpacker(f, raw=False)]
print(f"Frames loaded: {len(table_frames)}")

rvecs, tvecs = [], []
n_failed = 0

for i, frame_raw in enumerate(table_frames):
    frame  = np.array(frame_raw, dtype=np.uint8)
    # Handle both grayscale (1ch) and RGB (3ch) frames
    gray = frame[:HEIGHT, :WIDTH] 
    # gray   = cv2.flip(gray, 1)
    undist = cv2.remap(gray, map1, map2,
                       interpolation=cv2.INTER_LINEAR,
                       borderMode=cv2.BORDER_CONSTANT)

    charuco_corners, charuco_ids, _, _ = charuco_detector.detectBoard(undist)
    if charuco_corners is None or charuco_ids is None or len(charuco_ids) < 4:
        n_failed += 1
        continue

    # Same call as charuco_estimator.py — np.zeros((4,1)) matches original
    ok, rvec, tvec = cv2.aruco.estimatePoseCharucoBoard(
        charuco_corners, charuco_ids, board,
        new_K, np.zeros((4, 1)), None, None
    )
    if ok:
        rvecs.append(rvec.flatten())
        tvecs.append(tvec.flatten())
        if i % 50 == 0:
            print(f"  Frame {i:4d}: tvec={tvec.ravel().round(4)}")

print(f"Valid poses : {len(rvecs)} / {len(table_frames)}")
print(f"Failed      : {n_failed}")
assert len(rvecs) > 0, "No valid ChArUco pose — check board params and lighting"

# Average across all valid frames for stability
mean_rvec = np.mean(rvecs, axis=0).reshape(-1, 1)
mean_tvec = np.mean(tvecs, axis=0).reshape(-1, 1)
std_tvec  = np.std(tvecs,  axis=0)
R_cal, _  = cv2.Rodrigues(mean_rvec)

print(f"Mean tvec (m) : {mean_tvec.ravel().round(5)}")
print(f"tvec std  (m) : {std_tvec.round(5)}  (< 0.005 = stable)")
print(f"det(R)        : {np.linalg.det(R_cal):.6f}  ({'OK' if abs(np.linalg.det(R_cal)-1.0)<1e-4 else 'BAD'})")

# Save toml — same structure as charuco_estimator.py::save_calibration_data()
TABLE_CALIB_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with open(TABLE_CALIB_OUTPUT, "w") as f:
    toml.dump({
        "camera_matrix"  : cam_K.tolist(),
        "dist_coeffs"    : cam_dist.tolist(),
        "rvec"           : mean_rvec.tolist(),
        "tvec"           : mean_tvec.tolist(),
        "rotation_matrix": R_cal.tolist(),
        "n_frames_used"  : len(rvecs),
        "rvec_std"       : np.std(rvecs, axis=0).tolist(),
        "tvec_std"       : std_tvec.tolist(),
    }, f)
print(f"Saved -> {TABLE_CALIB_OUTPUT}")
print("Run Cell 5 to process experiment frames.")
frame0 = np.array(table_frames[0], dtype=np.uint8)
print(f"Frame shape: {frame0.shape}  dtype: {frame0.dtype}")

In [ ]:
# Cell 5 — PROCESS EXPERIMENT FRAMES (run per recording)
# Reads webcam_color.msgpack + webcam_timestamp.msgpack
# Detects NOARK ArUco markers per frame, saves tvec/rvec per marker
# Output: camera_data.csv with exact same columns as live recording CSV
# so Ground_truth_comparison_v2.ipynb works without any changes

# # ── Load table calibration ────────────────────────────────────────────────────
# tbl     = toml.load(TABLE_CALIB_OUTPUT)
# R_table = np.array(tbl["rotation_matrix"]).reshape(3, 3)
# T_table = np.array(tbl["tvec"]).reshape(3, 1)
# print(f"Table calibration loaded from {TABLE_CALIB_OUTPUT}")

# ── Load timestamps ───────────────────────────────────────────────────────────
print("Loading timestamps...")
with open(TIMESTAMPS_MSGPACK, "rb") as f:
    raw_ts = [item for item in msgpack.Unpacker(f, raw=False)]
# Each entry: [sync_pin (0/1), timestamp_string]
sync_pins  = [int(item[0]) for item in raw_ts]
timestamps = [datetime.fromisoformat(item[1]) for item in raw_ts]
print(f"Timestamps : {len(timestamps)}")
print(f"Sync HIGH  : {sum(sync_pins)} frames")
print(f"Duration   : {(timestamps[-1]-timestamps[0]).total_seconds():.2f} s")

# ── Load frames ───────────────────────────────────────────────────────────────
print("Loading frames...")
with open(FRAMES_MSGPACK, "rb") as f:
    raw_frames = [item for item in msgpack.Unpacker(f, raw=False)]
print(f"Frames loaded: {len(raw_frames)}")
assert len(raw_frames) == len(timestamps), (
    f"Frame/timestamp mismatch: {len(raw_frames)} vs {len(timestamps)}")

# ── Per-frame marker detection ────────────────────────────────────────────────
half     = MARKER_LENGTH_NOARK / 2.0
obj_pts  = np.array([[-half,  half, 0],
                     [ half,  half, 0],
                     [ half, -half, 0],
                     [-half, -half, 0]], dtype=np.float32)
no_dist  = np.zeros((5, 1), dtype=np.float64)

rows = []
for i, (frame_raw, ts, sync) in enumerate(zip(raw_frames, timestamps, sync_pins)):
    if i % 500 == 0:
        print(f"  Frame {i}/{len(raw_frames)}  ({100*i/len(raw_frames):.0f}%)")

    frame  = np.array(frame_raw, dtype=np.uint8)
    gray = frame[:HEIGHT, :WIDTH]
    # gray   = cv2.flip(gray, 1)                        # mirrors charuco_estimator.py
    undist = cv2.remap(gray, map1, map2,
                       interpolation=cv2.INTER_LINEAR,
                       borderMode=cv2.BORDER_CONSTANT)

    corners, ids, _ = aruco_detector.detectMarkers(undist)

    row = {"timestamp": ts.isoformat(), "sync_pin": sync}

    # initialise all marker columns as NaN
    for mid in MARKER_IDS:
        for ax in ["x", "y", "z"]:
            row[f"tvec_{mid}_{ax}"] = float("nan")
            row[f"rvec_{mid}_{ax}"] = float("nan")

    if ids is not None and len(ids) > 0:
        for corner, mid_arr in zip(corners, ids):
            mid = int(mid_arr[0])
            if mid not in MARKER_IDS:
                continue
            img_pts = corner[0].astype(np.float32)
            ok, rvec, tvec = cv2.solvePnP(
                obj_pts, img_pts, new_K, None,
                flags=cv2.SOLVEPNP_IPPE_SQUARE
            )
            if ok:
                tvec = tvec.ravel()   # (3,1) → (3,)
                rvec = rvec.ravel()   # (3,1) → (3,)
                row[f"tvec_{mid}_x"] = float(tvec[0])
                row[f"tvec_{mid}_y"] = float(tvec[1])
                row[f"tvec_{mid}_z"] = float(tvec[2])
                row[f"rvec_{mid}_x"] = float(rvec[0])
                row[f"rvec_{mid}_y"] = float(rvec[1])
                row[f"rvec_{mid}_z"] = float(rvec[2])
    rows.append(row)

# ── Save to CSV ───────────────────────────────────────────────────────────────
import pandas as pd
cam_df = pd.DataFrame(rows)
cam_df.to_csv(CAM_CSV_OUT, index=False)

valid = cam_df["tvec_12_x"].notna().sum()
print(f"\nSaved -> {CAM_CSV_OUT}")
print(f"Total frames  : {len(cam_df)}")
print(f"Valid marker 12 detections: {valid} ({100*valid/len(cam_df):.1f}%)")
print(f"Columns: {cam_df.columns.tolist()}")
print("\nDone. Now open Ground_truth_comparison_v2.ipynb and update CAM_CSV to point to this file.")


In [ ]:
# Cell 5 — PROCESS EXPERIMENT FRAMES + SAVE DETECTION VIDEO
import cv2
import msgpack
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# ── Paths ─────────────────────────────────────────────────────────────────────
RECORDING_DIR      = Path("E:/data/noark_data_may_15/linear_t1_may_15")
FRAMES_MSGPACK     = RECORDING_DIR / "webcam_color.msgpack"
TIMESTAMPS_MSGPACK = RECORDING_DIR / "webcam_timestamp.msgpack"
CAM_CSV_OUT        = RECORDING_DIR / "camera_data.csv"
VIDEO_OUT          = RECORDING_DIR / "detected_markers.mp4"

FPS    = 100
WIDTH  = 1280
HEIGHT = 800

# ── Load timestamps ───────────────────────────────────────────────────────────
print("Loading timestamps...")
with open(TIMESTAMPS_MSGPACK, "rb") as f:
    raw_ts = [item for item in msgpack.Unpacker(f, raw=False)]

sync_pins  = [int(item[0]) for item in raw_ts]
timestamps = [datetime.fromisoformat(item[1]) for item in raw_ts]

print(f"Timestamps : {len(timestamps)}")
print(f"Sync HIGH  : {sum(sync_pins)} frames")
print(f"Duration   : {(timestamps[-1] - timestamps[0]).total_seconds():.2f} s")

# ── Marker object points ──────────────────────────────────────────────────────
half    = MARKER_LENGTH_NOARK / 2.0
obj_pts = np.array([
    [-half,  half, 0],
    [ half,  half, 0],
    [ half, -half, 0],
    [-half, -half, 0]
], dtype=np.float32)

# ── Video writer ──────────────────────────────────────────────────────────────
fourcc    = cv2.VideoWriter_fourcc(*'mp4v')
video_out = cv2.VideoWriter(str(VIDEO_OUT), fourcc, FPS, (WIDTH, HEIGHT), isColor=True)
print(f"\nDetection video will be saved to: {VIDEO_OUT}")

# ── Process frames — streaming, no MemoryError ────────────────────────────────
rows             = []
detected_count   = 0
total_count      = 0

print("\nProcessing frames (streaming)...")

with open(FRAMES_MSGPACK, "rb") as f:
    unpacker = msgpack.Unpacker(f, raw=True, max_buffer_size=0)

    for i, frame_raw in enumerate(unpacker):

        if i >= len(timestamps):
            break

        ts   = timestamps[i]
        sync = sync_pins[i]
        total_count += 1

        if i % 500 == 0:
            print(f"  Frame {i}/{len(timestamps)}  ({100*i/len(timestamps):.0f}%)  "
                  f"detected so far: {detected_count}")

        # ── Decode frame ──────────────────────────────────────────────────────
        if i == 0:
            print(f"\nframe_raw type   : {type(frame_raw)}")
            print(f"frame_raw length : {len(frame_raw)}\n")

        try:
            if isinstance(frame_raw, bytes):
                gray = np.frombuffer(frame_raw, dtype=np.uint8).reshape(HEIGHT, WIDTH)
            elif isinstance(frame_raw, (list, tuple)):
                gray = np.array(frame_raw, dtype=np.uint8).reshape(HEIGHT, WIDTH)
            elif isinstance(frame_raw, dict):
                data = frame_raw.get(b'data') or frame_raw.get('data')
                gray = np.frombuffer(data, dtype=np.uint8).reshape(HEIGHT, WIDTH)
            else:
                print(f"Unknown frame type at {i}: {type(frame_raw)}")
                continue
        except Exception as e:
            print(f"Frame {i} decode error: {e}")
            continue

        # ── Undistort ─────────────────────────────────────────────────────────
        undist = cv2.remap(gray, map1, map2,
                           interpolation=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_CONSTANT)

        # ── Detect markers ────────────────────────────────────────────────────
        corners, ids, _ = aruco_detector.detectMarkers(undist)

        # ── CSV row ───────────────────────────────────────────────────────────
        row = {"timestamp": ts.isoformat(), "sync_pin": sync}
        for mid in MARKER_IDS:
            for ax in ["x", "y", "z"]:
                row[f"tvec_{mid}_{ax}"] = float("nan")
                row[f"rvec_{mid}_{ax}"] = float("nan")

        # ── Only process and write frames where markers are detected ──────────
        marker_detected = False

        if ids is not None and len(ids) > 0:

            vis = cv2.cvtColor(undist.copy(), cv2.COLOR_GRAY2BGR)
            cv2.putText(vis, f"Frame: {i}", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            cv2.aruco.drawDetectedMarkers(vis, corners, ids)

            for corner, mid_arr in zip(corners, ids):
                mid = int(mid_arr[0])
                if mid not in MARKER_IDS:
                    continue

                img_pts = corner[0].astype(np.float32)
                ok, rvec, tvec = cv2.solvePnP(
                    obj_pts, img_pts, new_K, None,
                    flags=cv2.SOLVEPNP_IPPE_SQUARE
                )

                if ok:
                    tvec = tvec.ravel()
                    rvec = rvec.ravel()

                    row[f"tvec_{mid}_x"] = float(tvec[0])
                    row[f"tvec_{mid}_y"] = float(tvec[1])
                    row[f"tvec_{mid}_z"] = float(tvec[2])
                    row[f"rvec_{mid}_x"] = float(rvec[0])
                    row[f"rvec_{mid}_y"] = float(rvec[1])
                    row[f"rvec_{mid}_z"] = float(rvec[2])

                    cv2.drawFrameAxes(vis, new_K, None,
                                      rvec.reshape(3, 1), tvec.reshape(3, 1),
                                      MARKER_LENGTH_NOARK * 0.5, 2)

                    c   = np.mean(img_pts, axis=0).astype(int)
                    cv2.putText(vis, f"ID {mid}", (c[0] + 10, c[1]),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

                    marker_detected = True

            # ── Write ONLY detected frames to video ───────────────────────────
            video_out.write(vis)
            detected_count += 1

        rows.append(row)

# ── Release ───────────────────────────────────────────────────────────────────
video_out.release()

# ── Save CSV ──────────────────────────────────────────────────────────────────
cam_df = pd.DataFrame(rows)
cam_df.to_csv(CAM_CSV_OUT, index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"Total frames processed    : {total_count}")
print(f"Frames written to video   : {detected_count}  ({100*detected_count/total_count:.1f}%)")
print(f"Frames with no detection  : {total_count - detected_count}  ({100*(total_count-detected_count)/total_count:.1f}%)")
print(f"\nPer-marker detection rates:")
for mid in MARKER_IDS:
    valid = cam_df[f"tvec_{mid}_x"].notna().sum()
    print(f"  Marker {mid}: {valid} / {total_count}  ({100*valid/total_count:.1f}%)")
print(f"\nSaved CSV   -> {CAM_CSV_OUT}")
print(f"Saved video -> {VIDEO_OUT}")